In [ ]:
from pathlib import Path
import anndata as ad
import sys

project_root = Path.cwd().parent
sys.path.append(str(project_root / "src"))

import argparse
from datetime import datetime

from methylvae.utils.config import load_config, merge_configs_with_search_space
from methylvae.training.train import train

%load_ext autoreload
%autoreload 2

In [ ]:
adata = ad.read_h5ad("/Users/sophiali/Desktop/MethylVAE/data/pancancer_v2_test_adata.h5ad")

In [ ]:
import numpy as np
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("DataIntegrity")

def verify_data_integrity(adata):
    """
    Assesses integrity of Z-score normalized DNA methylation data.
    """
    logger.info("=====| Starting Data Integrity Assessment |=====")
    X = np.asarray(adata.X)
    
    # 1. NaN and Inf Check
    if np.isnan(X).any() or np.isinf(X).any():
        logger.error("Integrity FAILED: Data contains NaNs or Infs.")
    else:
        logger.info("Integrity PASSED: No NaNs or Infs detected.")

    # 2. Scaling Check (Z-score should result in Mean ~0 and Std ~1)
    # We calculate feature-wise stats
    col_means = np.mean(X, axis=0)
    col_stds = np.std(X, axis=0)
    
    mean_of_means = np.mean(col_means)
    mean_of_stds = np.mean(col_stds)
    
    if np.isclose(mean_of_means, 0, atol=1e-1) and np.isclose(mean_of_stds, 1, atol=1e-1):
        logger.info(f"Scaling PASSED: Mean={mean_of_means:.4f}, Std={mean_of_stds:.4f}")
    else:
        logger.warning(f"Scaling SUSPICIOUS: Mean={mean_of_means:.4f}, Std={mean_of_stds:.4f}")
        logger.warning("Check if normalization was actually applied or if data is skewed.")

    # 3. Winsorization Check (Min/Max values)
    # With Z-score, 99.9% of data should typically fall within [-5, 5]
    abs_max = np.max(np.abs(X))
    if abs_max > 10:
        logger.warning(f"Outlier Alert: Max absolute value is {abs_max:.2f}. "
                       "Winsorization may not be sufficiently aggressive.")
    else:
        logger.info(f"Outlier Check PASSED: Max absolute value is {abs_max:.2f}")

    # 4. Data Type Check
    if X.dtype != np.float32:
        logger.warning(f"Data type is {X.dtype}, expected float32. Consider casting for VAE.")
    else:
        logger.info("Data type PASSED: float32 confirmed.")

    logger.info("=====| Data Integrity Check Complete |=====")

# Run the check
verify_data_integrity(adata)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Pick 5 random column indices
indices = np.random.choice(adata.shape[1], 5, replace=False)
plt.figure(figsize=(10, 5))
for i in indices:
    sns.kdeplot(adata.X[:, i], label=f'Probe {i}')
plt.title("Distribution of 5 Random Probes")
plt.legend()
plt.show()

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
coords = pca.fit_transform(adata.X)

plt.figure(figsize=(8, 6))
# Replace 'cancer_type' with your actual metadata column
plt.scatter(coords[:, 0], coords[:, 1], c=adata.obs['project_id'].astype('category').cat.codes, s=5, cmap='viridis')
plt.title("PCA of 30k Probes")
plt.show()

In [ ]:
# Sample 100 probes
subset = adata.X[:, np.random.choice(adata.shape[1], 100, replace=False)]
corr = np.corrcoef(subset, rowvar=False)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, cmap='RdBu_r', center=0)
plt.title("Correlation of 100 Random Probes")
plt.show()